# Weather -> Renewable Generation: a Bayesian predictive model

This notebook fits Bayesian regressions connecting local weather to German renewable generation, following the correct Bayesian workflow order: EDA on the marginal distributions first, likelihood choice from that EDA (not assumed up front), a prior predictive check before the model ever sees data, then fit -> convergence diagnostics -> posterior predictive check -> formal model comparison against a Normal-likelihood baseline via PSIS-LOO.

- `wind_onshore_mw ~ wind_speed_kmh` -- Gamma GLM, log link (wind is strictly positive, never zero, right-skewed)
- `solar_mw ~ temperature_c + cloud_cover_pct` -- hurdle-Gamma (solar has a genuine zero-vs-positive two-part structure: night hours read exactly 0, daylight hours are a separate right-skewed process)

Same evaluation logic throughout:

1. **Best case** -- predict using the *actual* (historical) weather for the same hours used to fit the model. This is the error floor: the best this simple model could ever do, even with perfect weather knowledge.
2. **Forecast-driven** -- predict using the *forecast* weather (Open-Meteo) for those same hours. This is what you'd actually achieve day-to-day.

The gap between (2) and (1) is the accuracy cost attributable specifically to weather forecast error. Every prediction comes from a posterior sample of the model's parameters, so that gap comes out as a distribution -- e.g. "MAE gap: +372 MW (94% CI: +310, +436)" -- not a single number that hides how much of it is genuine signal versus sampling noise.

We finish with an imbalance-equivalent cost, price-weighted by the real day-ahead price for each hour if `data/raw/energy_prices.parquet` exists (produced by `src/ingest_price.py`) -- still clearly labeled as illustrative, not a real settlement/trading estimate, since real imbalance settlement uses signed (not absolute) error against a distinct imbalance price, not the day-ahead price.

In [2]:
from __future__ import annotations

import os

# This venv's Python is the Windows Store build (WindowsApps package dir);
# PyTensor's default C-compiled backend fails to link against its python3XX.dll
# there (permission-restricted package directory), so PyMC would otherwise
# error out before sampling ever starts. Forcing the pure-Python linker fixes
# that at the cost of speed -- if you're running this on a "normal"
# (python.org / conda) Python install where `pytensor` can compile fine,
# delete this line for a large speedup (compiled NUTS gradients instead of
# interpreted ones).
os.environ.setdefault("PYTENSOR_FLAGS", "cxx=")

from pathlib import Path

import arviz as az
import matplotlib
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt

matplotlib.use("Agg")  # headless: this notebook only writes files, no display needed
import matplotlib.pyplot as plt

# Notebook lives in notebooks/, project root is one level up. Robust to being
# run from either directory (Jupyter's default cwd is the notebook's own dir).
if (Path.cwd() / "data").exists():
    PROJECT_ROOT = Path.cwd()
elif (Path.cwd().parent / "data").exists():
    PROJECT_ROOT = Path.cwd().parent
else:
    raise FileNotFoundError(
        "Could not locate the project's data/ directory relative to the current "
        f"working directory ({Path.cwd()}). Run this notebook from the project "
        "root or from notebooks/."
    )

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# MCMC settings. 4 chains / 1000 tune / 1000 draws is PyMC's own default and
# gives well-resolved R-hat/ESS diagnostics; on a machine stuck on the
# pure-Python linker (see above) this is slow -- drop to e.g. N_CHAINS=2,
# N_TUNE=300, N_DRAWS=300 if you just want to confirm the notebook runs
# end-to-end.
N_CHAINS = 2
N_TUNE = 500
N_DRAWS = 500
RANDOM_SEED = 42
HDI_PROB = 0.94  # ArviZ's default credible-interval width

print(f"[generation_model] PROJECT_ROOT = {PROJECT_ROOT}")
print(f"[generation_model] PyMC {pm.__version__}, ArviZ {az.__version__}, MCMC: {N_CHAINS} chains x ({N_TUNE} tune + {N_DRAWS} draws)")

d:\weather-energy-analytics\.venv\Lib\site-packages\arviz\__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


[generation_model] PROJECT_ROOT = d:\weather-energy-analytics
[generation_model] PyMC 5.28.5, ArviZ 0.23.4, MCMC: 2 chains x (500 tune + 500 draws)


## 1. Load data

- `data/processed/weather_energy_joined.parquet` -- actual weather inner-joined with actual energy generation (produced by `src/analysis.py`).
- `data/raw/weather_forecast.parquet` -- Open-Meteo forecast weather (produced by `src/ingest_weather.py`).
- `data/raw/energy_prices.parquet` -- day-ahead price (produced by `src/ingest_price.py`), used only in Section 9 for the price-weighted imbalance cost. Optional: if missing, that part is skipped with a clear message -- everything else in the notebook still runs.

Weather/energy inputs fail loudly if missing -- no fabricated fallback data.

In [3]:
def _require(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing input file: {path}\n"
            "Run the ingest/analysis scripts first (ingest_weather.py, ingest_energy.py, analysis.py)."
        )
    return pd.read_parquet(path)


print("[generation_model] Loading weather_energy_joined.parquet and weather_forecast.parquet...")
joined = _require(PROCESSED_DIR / "weather_energy_joined.parquet")
weather_fcst = _require(RAW_DIR / "weather_forecast.parquet")

print(f"[generation_model] joined:       {len(joined):,} rows, {joined['timestamp'].min()} -> {joined['timestamp'].max()}")
print(f"[generation_model] weather_fcst: {len(weather_fcst):,} rows, {weather_fcst['timestamp'].min()} -> {weather_fcst['timestamp'].max()}")

price_path = RAW_DIR / "energy_prices.parquet"
have_price = price_path.exists()
if have_price:
    prices = pd.read_parquet(price_path)[["timestamp", "price_eur_mwh"]]
    print(f"[generation_model] energy_prices: {len(prices):,} rows, {prices['timestamp'].min()} -> {prices['timestamp'].max()}")
else:
    prices = None
    print(f"[generation_model] {price_path} not found -- the price-weighted cost section will be skipped.")

joined.head()

[generation_model] Loading weather_energy_joined.parquet and weather_forecast.parquet...
[generation_model] joined:       2,157 rows, 2026-06-17 00:00:00 -> 2026-09-14 22:00:00
[generation_model] weather_fcst: 2,184 rows, 2026-06-16 00:00:00 -> 2026-09-14 23:00:00
[generation_model] energy_prices: 8,544 rows, 2026-06-17 09:30:00 -> 2026-09-15 09:15:00


,timestamp,temperature_c,wind_speed_kmh,cloud_cover_pct,total_load_mw,wind_onshore_mw,wind_offshore_mw,solar_mw,wind_total_mw
0,2026-06-17 00:00:00,16.3,3.1,100,40665.71126,2965.611652,315.823,23.624984,3281.434652
1,2026-06-17 01:00:00,15.9,3.8,99,40624.52763,2360.737300,359.648,24.806220,2720.385300
2,2026-06-17 02:00:00,15.6,7.2,91,41193.13678,1853.515440,546.471,22.456084,2399.986440
3,2026-06-17 03:00:00,15.3,9.3,98,43159.95808,1537.446296,580.352,227.138232,2117.798296
4,2026-06-17 04:00:00,15.6,8.9,98,48896.55673,1420.684160,880.402,2949.645208,2301.086160


## 2. Exploratory data analysis: marginal distributions (before any modeling)

Marginal histograms of `wind_onshore_mw` and `solar_mw` alone, no predictors -- the likelihood choice below is driven by what these actually look like, not assumed up front.

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(joined["wind_onshore_mw"], bins=40, color="#2c6e9e", edgecolor="white")
axes[0].set_title("wind_onshore_mw (marginal, no predictors)")
axes[0].set_xlabel("MW")

axes[1].hist(joined["solar_mw"], bins=40, color="#c0392b", edgecolor="white")
axes[1].set_title("solar_mw (marginal, no predictors)")
axes[1].set_xlabel("MW")
fig.tight_layout()

eda_plot_path = PROCESSED_DIR / "generation_model_eda_marginals.png"
fig.savefig(eda_plot_path, dpi=130)
plt.close(fig)
print(f"[generation_model] Saved plot -> {eda_plot_path}")

print()
for col in ["wind_onshore_mw", "solar_mw"]:
    s = joined[col]
    n_zero = int((s == 0).sum())
    print(f"[generation_model] {col}: n={len(s):,}, min={s.min():,.2f}, max={s.max():,.2f}, "
          f"skew={s.skew():.3f}, n_exact_zero={n_zero} ({n_zero / len(s):.1%})")

[generation_model] Saved plot -> d:\weather-energy-analytics\data\processed\generation_model_eda_marginals.png

[generation_model] wind_onshore_mw: n=2,157, min=72.38, max=40,573.35, skew=1.054, n_exact_zero=0 (0.0%)
[generation_model] solar_mw: n=2,157, min=0.00, max=58,506.98, skew=0.723, n_exact_zero=568 (26.3%)


## 3. Likelihood choice and priors (GLM framework)

**EDA findings** (computed above): `wind_onshore_mw` is strictly positive (never exactly zero) and right-skewed. `solar_mw` has a large exact-zero mass (overnight hours) plus a right-skewed continuous positive mass during daylight -- a genuinely two-part structure, not a single skewed distribution (the histogram matters more than any single skew statistic here).

**Likelihood choice**, GLM-style (the generalized-linear-model exponential-family framework: a linear predictor, a link function, and a distribution family -- generalizing the Normal-identity special case, i.e. ordinary linear regression, to whichever family actually matches the response's support and shape):

- **wind**: bounded at 0, never touches it, right-skewed -> **Gamma likelihood, log link**: `wind_onshore_mw ~ Gamma(mean=mu, shape=alpha)`, `log(mu) = b0 + beta . wind_speed_z`. This is the standard Gamma-regression recipe (`y ~ Gamma(mean=mu, shape=alpha)`, `mu = exp(b0 + sum bk*xk)`) for a strictly-positive, right-skewed response.
- **solar**: a point mass at exactly 0 plus a separate continuous process for the positive hours. In finite-mixture-model terms (`Y ~ w1*T(theta1) + w2*T(theta2)`), this is a degenerate two-component mixture where one component is a point mass at 0 -- the special case usually called a **hurdle model**: `P(Y=0) = 1-p`, `P(Y=y | y>0) = p * Gamma(y; mean=mu_pos, shape=alpha)`, with both `p` (logistic regression) and `mu_pos` (log-link Gamma regression) depending on `temperature_c` and `cloud_cover_pct`. Built as an explicit manual log-likelihood below (`pm.CustomDist` with a hand-written `logp`, via `pm.logp` on `pm.Gamma.dist(...)`), since brms/Stan's built-in `hurdle_gamma` family isn't part of this project's stack.

**Priors**: same *structure* as a standard Gamma-regression setup (Normal priors on the log-link coefficients, Gamma prior on the shape parameter) but rescaled to this data's actual magnitude, following a general scale-awareness principle for prior specification -- a naive `b0 ~ Normal(2, 5)` assumes a response near `exp(2) ~= 7.4`, nowhere close to our MW-scale response. Intercepts are instead centered at the log of each response's typical order of magnitude, with `sigma=1.0` (weakly informative: roughly 1-2 orders of magnitude around that center). Slope coefficients use `Normal(0, 1)` on standardized (z-scored) predictors -- scale-free by construction. The Gamma shape parameter uses `Gamma(2, 0.1)` (mean 20, matching the well-behaved shape-parameter prior used elsewhere in this project) rather than the diffuse textbook default `Gamma(0.01, 0.01)`, which is diffuse enough to put substantial prior mass near shape~=0 -- a pathological, near-degenerate corner of Gamma noise that the prior predictive check below is specifically designed to catch.

In [5]:
wind_x_cols = ["wind_speed_kmh"]
wind_sub = joined[wind_x_cols + ["wind_onshore_mw"]].dropna()
wind_X = wind_sub[wind_x_cols].to_numpy(dtype=float)
wind_y = wind_sub["wind_onshore_mw"].to_numpy(dtype=float)
wind_x_mean, wind_x_std = wind_X.mean(axis=0), wind_X.std(axis=0)
wind_Xz = (wind_X - wind_x_mean) / wind_x_std

solar_x_cols = ["temperature_c", "cloud_cover_pct"]
solar_sub = joined[solar_x_cols + ["solar_mw"]].dropna()
solar_X = solar_sub[solar_x_cols].to_numpy(dtype=float)
solar_y = solar_sub["solar_mw"].to_numpy(dtype=float)
solar_x_mean, solar_x_std = solar_X.mean(axis=0), solar_X.std(axis=0)
solar_Xz = (solar_X - solar_x_mean) / solar_x_std

WIND_B0_PRIOR_MU = float(np.log(wind_y.mean()))
SOLAR_B0_PRIOR_MU = float(np.log(solar_y[solar_y > 0].mean()))
SHAPE_PRIOR_ALPHA, SHAPE_PRIOR_BETA = 2.0, 0.1  # weakly-informative Gamma(shape) prior; see markdown above

print(f"[generation_model] wind: b0 prior center = log(mean(wind_onshore_mw)) = {WIND_B0_PRIOR_MU:.2f}")
print(f"[generation_model] solar: b0 prior center = log(mean(solar_mw | solar_mw>0)) = {SOLAR_B0_PRIOR_MU:.2f}")


def build_wind_gamma_model(Xz: np.ndarray, y=None) -> pm.Model:
    """Gamma GLM, log link: wind_onshore_mw ~ Gamma(mean=mu, shape=alpha),
    log(mu) = b0 + beta . Xz. y=None builds an unconditioned model (prior
    predictive / prior-only exploration); y=wind_y conditions on the data."""
    with pm.Model(coords={"predictor": wind_x_cols}) as model:
        b0 = pm.Normal("b0", mu=WIND_B0_PRIOR_MU, sigma=1.0)
        beta = pm.Normal("beta", mu=0, sigma=1, dims="predictor")
        alpha = pm.Gamma("alpha", alpha=SHAPE_PRIOR_ALPHA, beta=SHAPE_PRIOR_BETA)
        mu = pm.Deterministic("mu", pm.math.exp(b0 + pm.math.dot(Xz, beta)))
        if y is None:
            pm.Gamma("y_obs", alpha=alpha, beta=alpha / mu, shape=Xz.shape[0])
        else:
            pm.Gamma("y_obs", alpha=alpha, beta=alpha / mu, observed=y)
    return model


def hurdle_gamma_logp(value, p, mu, alpha):
    """Pointwise log-density of the hurdle-Gamma: P(Y=0)=1-p,
    P(Y=y|y>0) = p * Gamma(y; mean=mu, shape=alpha). Built manually via
    pm.logp on pm.Gamma.dist (not brms/Stan's hurdle_gamma family)."""
    safe_value = pt.switch(pt.eq(value, 0), 1.0, value)  # dummy value where the Gamma branch is unused
    gamma_lp = pm.logp(pm.Gamma.dist(alpha=alpha, beta=alpha / mu), safe_value)
    return pt.switch(pt.eq(value, 0), pt.log1p(-p), pt.log(p) + gamma_lp)


def build_solar_hurdle_model(Xz: np.ndarray, y) -> pm.Model:
    """Hurdle-Gamma, both parts on the same predictors:
    P(nonzero) = invlogit(g0 + gamma_beta . Xz)   (Bernoulli part)
    mean(y | y>0) = exp(b0 + beta . Xz)           (Gamma part, log link)"""
    with pm.Model(coords={"predictor": solar_x_cols}) as model:
        g0 = pm.Normal("g0", mu=0, sigma=1.0)
        gamma_beta = pm.Normal("gamma_beta", mu=0, sigma=1, dims="predictor")
        p_nonzero = pm.Deterministic("p_nonzero", pm.math.sigmoid(g0 + pm.math.dot(Xz, gamma_beta)))

        b0 = pm.Normal("b0", mu=SOLAR_B0_PRIOR_MU, sigma=1.0)
        beta = pm.Normal("beta", mu=0, sigma=1, dims="predictor")
        mu_positive = pm.Deterministic("mu_positive", pm.math.exp(b0 + pm.math.dot(Xz, beta)))

        alpha = pm.Gamma("alpha", alpha=SHAPE_PRIOR_ALPHA, beta=SHAPE_PRIOR_BETA)

        pm.CustomDist(
            "y_obs", p_nonzero, mu_positive, alpha,
            logp=hurdle_gamma_logp,
            observed=y,
        )
    return model


def build_solar_prior_only_model(Xz: np.ndarray) -> pm.Model:
    """Same priors/deterministics as build_solar_hurdle_model, without the
    CustomDist likelihood node -- used only for the prior predictive check
    below, where y is simulated manually in numpy rather than via PyMC's own
    prior-predictive machinery (the hurdle CustomDist has no `random=`,
    since only its logp is needed for fitting and for LOO)."""
    with pm.Model(coords={"predictor": solar_x_cols}) as model:
        g0 = pm.Normal("g0", mu=0, sigma=1.0)
        gamma_beta = pm.Normal("gamma_beta", mu=0, sigma=1, dims="predictor")
        pm.Deterministic("p_nonzero", pm.math.sigmoid(g0 + pm.math.dot(Xz, gamma_beta)))

        b0 = pm.Normal("b0", mu=SOLAR_B0_PRIOR_MU, sigma=1.0)
        beta = pm.Normal("beta", mu=0, sigma=1, dims="predictor")
        pm.Deterministic("mu_positive", pm.math.exp(b0 + pm.math.dot(Xz, beta)))

        pm.Gamma("alpha", alpha=SHAPE_PRIOR_ALPHA, beta=SHAPE_PRIOR_BETA)
    return model

[generation_model] wind: b0 prior center = log(mean(wind_onshore_mw)) = 9.20
[generation_model] solar: b0 prior center = log(mean(solar_mw | solar_mw>0)) = 9.95


## 4. Prior predictive check

Simulate from the priors alone -- no data -- and confirm the simulated `wind_onshore_mw` / `solar_mw` values are non-negative and the right order of magnitude before the model ever sees the real data. This is the step a naive Normal-likelihood approach would skip.

In [6]:
N_PRIOR_DRAWS = 1000

with build_wind_gamma_model(wind_Xz, y=None) as wind_prior_model:
    wind_prior_idata = pm.sample_prior_predictive(samples=N_PRIOR_DRAWS, random_seed=RANDOM_SEED)
wind_prior_y = wind_prior_idata.prior["y_obs"].values.flatten()

with build_solar_prior_only_model(solar_Xz) as solar_prior_model:
    solar_prior_idata = pm.sample_prior_predictive(samples=N_PRIOR_DRAWS, random_seed=RANDOM_SEED)

rng = np.random.default_rng(RANDOM_SEED)
prior_p = solar_prior_idata.prior["p_nonzero"].values             # (chain, draw, N)
prior_mu = solar_prior_idata.prior["mu_positive"].values          # (chain, draw, N)
prior_alpha = np.broadcast_to(solar_prior_idata.prior["alpha"].values[..., None], prior_mu.shape)

nonzero_draw = rng.uniform(size=prior_p.shape) < prior_p
gamma_draws = rng.gamma(shape=prior_alpha, scale=prior_mu / prior_alpha)
solar_prior_y = np.where(nonzero_draw, gamma_draws, 0.0).flatten()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(np.clip(wind_prior_y, 0, wind_y.max() * 3), bins=50, color="#2c6e9e")
axes[0].axvline(wind_y.max(), color="black", linestyle="--", label="observed max")
axes[0].set_title("Prior predictive: wind_onshore_mw")
axes[0].set_xlabel("MW (clipped at 3x observed max for plotting)")
axes[0].legend()

axes[1].hist(np.clip(solar_prior_y, 0, solar_y.max() * 3), bins=50, color="#c0392b")
axes[1].axvline(solar_y.max(), color="black", linestyle="--", label="observed max")
axes[1].set_title("Prior predictive: solar_mw")
axes[1].set_xlabel("MW (clipped at 3x observed max for plotting)")
axes[1].legend()
fig.tight_layout()

prior_pred_plot_path = PROCESSED_DIR / "generation_model_prior_predictive.png"
fig.savefig(prior_pred_plot_path, dpi=130)
plt.close(fig)
print(f"[generation_model] Saved plot -> {prior_pred_plot_path}")

print()
for name, draws, y_obs_ref in [("wind_onshore_mw", wind_prior_y, wind_y), ("solar_mw", solar_prior_y, solar_y)]:
    frac_negative = float((draws < 0).mean())
    frac_within_10x = float(((draws >= 0) & (draws <= 10 * y_obs_ref.max())).mean())
    print(f"[generation_model] {name} prior predictive: "
          f"min={draws.min():,.1f}, median={np.median(draws):,.1f}, max={draws.max():,.1f}, "
          f"frac_negative={frac_negative:.4f} (must be 0.0 -- Gamma/hurdle support is y>=0), "
          f"frac_within_10x_observed_max={frac_within_10x:.3f}")

frac_zero_prior = float((solar_prior_y == 0).mean())
print(f"[generation_model] solar_mw prior predictive zero-mass: {frac_zero_prior:.1%} "
      f"(observed: {(solar_y == 0).mean():.1%}) -- check this is in a plausible ballpark before fitting.")

Sampling: [alpha, b0, beta, y_obs]
Sampling: [alpha, b0, beta, g0, gamma_beta]


[generation_model] Saved plot -> d:\weather-energy-analytics\data\processed\generation_model_prior_predictive.png

[generation_model] wind_onshore_mw prior predictive: min=0.0, median=9,895.2, max=2,102,073,067.6, frac_negative=0.0000 (must be 0.0 -- Gamma/hurdle support is y>=0), frac_within_10x_observed_max=0.991
[generation_model] solar_mw prior predictive: min=0.0, median=325.8, max=438,968,432.1, frac_negative=0.0000 (must be 0.0 -- Gamma/hurdle support is y>=0), frac_within_10x_observed_max=0.986
[generation_model] solar_mw prior predictive zero-mass: 49.4% (observed: 26.3%) -- check this is in a plausible ballpark before fitting.


## 5. Fit the models

In [7]:
print("[generation_model] Fitting wind Gamma GLM (log link)...")
with build_wind_gamma_model(wind_Xz, y=wind_y) as wind_gamma_model:
    wind_gamma_idata = pm.sample(
        draws=N_DRAWS, tune=N_TUNE, chains=N_CHAINS,
        target_accept=0.9, random_seed=RANDOM_SEED, progressbar=False,
    )
wind_gamma_fit = {
    "model": wind_gamma_model, "idata": wind_gamma_idata, "sub": wind_sub,
    "x_cols": wind_x_cols, "y_col": "wind_onshore_mw",
    "x_mean": wind_x_mean, "x_std": wind_x_std,
}

print("[generation_model] Fitting solar hurdle-Gamma...")
with build_solar_hurdle_model(solar_Xz, y=solar_y) as solar_hurdle_model:
    solar_hurdle_idata = pm.sample(
        draws=N_DRAWS, tune=N_TUNE, chains=N_CHAINS,
        target_accept=0.9, random_seed=RANDOM_SEED, progressbar=False,
    )
solar_hurdle_fit = {
    "model": solar_hurdle_model, "idata": solar_hurdle_idata, "sub": solar_sub,
    "x_cols": solar_x_cols, "y_col": "solar_mw",
    "x_mean": solar_x_mean, "x_std": solar_x_std,
}
print("[generation_model] Both models fit.")

[generation_model] Fitting wind Gamma GLM (log link)...


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [b0, beta, alpha]
Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 2495 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


[generation_model] Fitting solar hurdle-Gamma...


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [g0, gamma_beta, b0, beta, alpha]
Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 3144 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


[generation_model] Both models fit.


## 6. Convergence diagnostics

R-hat should be within ~0.01 of 1.00 and effective sample size (bulk/tail) should be a reasonable fraction of the total post-warmup draws for the posterior to be trusted; 0 divergences means NUTS could fully explore the posterior.

In [8]:
wind_gamma_summary = az.summary(wind_gamma_fit["idata"], var_names=["b0", "beta", "alpha"], hdi_prob=HDI_PROB)
solar_hurdle_summary = az.summary(
    solar_hurdle_fit["idata"], var_names=["g0", "gamma_beta", "b0", "beta", "alpha"], hdi_prob=HDI_PROB
)

print("[generation_model] wind Gamma GLM diagnostics:")
print(wind_gamma_summary)
print("\n[generation_model] solar hurdle-Gamma diagnostics:")
print(solar_hurdle_summary)

diag_ok = True
for name, summary in [("wind_gamma", wind_gamma_summary), ("solar_hurdle", solar_hurdle_summary)]:
    bad_rhat = summary[(summary["r_hat"] - 1.0).abs() > 0.01]
    low_ess = summary[summary["ess_bulk"] < 400]
    if len(bad_rhat) or len(low_ess):
        diag_ok = False
        print(f"[generation_model] WARNING ({name}): R-hat/ESS outside comfortable bounds -- see rows above.")

print("\n[generation_model] Divergent transitions per chain:")
for name, fit in [("wind_gamma", wind_gamma_fit), ("solar_hurdle", solar_hurdle_fit)]:
    per_chain = fit["idata"].sample_stats["diverging"].sum(dim="draw").values
    total = int(per_chain.sum())
    chain_str = ", ".join(f"chain {i}: {int(d)}" for i, d in enumerate(per_chain))
    print(f"[generation_model]   {name}: total={total} ({chain_str})")
    if total:
        diag_ok = False

if diag_ok:
    print("\n[generation_model] All R-hat within 0.01 of 1.00, ESS >= 400, 0 divergences.")

[generation_model] wind Gamma GLM diagnostics:
                       mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  \
b0                    9.102  0.014   9.076    9.129      0.000    0.000   
beta[wind_speed_kmh]  0.432  0.014   0.408    0.460      0.000    0.000   
alpha                 2.378  0.067   2.244    2.496      0.002    0.002   

                      ess_bulk  ess_tail  r_hat  
b0                       863.0     694.0    1.0  
beta[wind_speed_kmh]     867.0     764.0    1.0  
alpha                   1126.0     762.0    1.0  

[generation_model] solar hurdle-Gamma diagnostics:
                              mean     sd  hdi_3%  hdi_97%  mcse_mean  \
g0                           1.367  0.065   1.250    1.490      0.002   
gamma_beta[temperature_c]    1.292  0.077   1.144    1.426      0.002   
gamma_beta[cloud_cover_pct]  0.266  0.055   0.163    0.372      0.002   
b0                           9.798  0.038   9.728    9.868      0.001   
beta[temperature_c]          0.374 

### 6b. Trace plots (chain mixing)

In [9]:
axes = az.plot_trace(wind_gamma_fit["idata"], var_names=["b0", "beta", "alpha"], compact=False, figsize=(10, 6))
fig = np.asarray(axes).ravel()[0].figure
fig.suptitle("Trace: wind Gamma GLM", y=1.01)
fig.tight_layout()
trace_wind_path = PROCESSED_DIR / "generation_model_trace_wind.png"
fig.savefig(trace_wind_path, dpi=130)
plt.close(fig)
print(f"[generation_model] Saved plot -> {trace_wind_path}")

axes = az.plot_trace(
    solar_hurdle_fit["idata"], var_names=["g0", "gamma_beta", "b0", "beta", "alpha"], compact=False, figsize=(10, 12)
)
fig = np.asarray(axes).ravel()[0].figure
fig.suptitle("Trace: solar hurdle-Gamma", y=1.01)
fig.tight_layout()
trace_solar_path = PROCESSED_DIR / "generation_model_trace_solar.png"
fig.savefig(trace_solar_path, dpi=130)
plt.close(fig)
print(f"[generation_model] Saved plot -> {trace_solar_path}")

[generation_model] Saved plot -> d:\weather-energy-analytics\data\processed\generation_model_trace_wind.png
[generation_model] Saved plot -> d:\weather-energy-analytics\data\processed\generation_model_trace_solar.png


## 7. Posterior predictive check

Confirm the new likelihoods actually capture the shape this time -- solar's zero spike, wind's right skew.

In [10]:
with wind_gamma_fit["model"]:
    wind_gamma_ppc = pm.sample_posterior_predictive(
        wind_gamma_fit["idata"], var_names=["y_obs"], random_seed=RANDOM_SEED, progressbar=False,
    )
wind_gamma_ppc_idata = az.from_dict(
    posterior_predictive={"wind_onshore_mw": wind_gamma_ppc.posterior_predictive["y_obs"].values},
    observed_data={"wind_onshore_mw": wind_y},
)

# Solar hurdle has no `random=` (logp-only CustomDist, since only its logp is needed for fitting
# and for LOO) -- simulate manually from the posterior draws of p_nonzero / mu_positive / alpha
# instead, using the same hurdle mechanics as the model's own logp.
post_p = solar_hurdle_fit["idata"].posterior["p_nonzero"].values
post_mu = solar_hurdle_fit["idata"].posterior["mu_positive"].values
post_alpha = np.broadcast_to(solar_hurdle_fit["idata"].posterior["alpha"].values[..., None], post_mu.shape)

rng = np.random.default_rng(RANDOM_SEED)
nonzero_draw = rng.uniform(size=post_p.shape) < post_p
gamma_draws = rng.gamma(shape=post_alpha, scale=post_mu / post_alpha)
solar_hurdle_ppc_y = np.where(nonzero_draw, gamma_draws, 0.0)

solar_hurdle_ppc_idata = az.from_dict(
    posterior_predictive={"solar_mw": solar_hurdle_ppc_y},
    observed_data={"solar_mw": solar_y},
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
az.plot_ppc(wind_gamma_ppc_idata, var_names=["wind_onshore_mw"], ax=axes[0], num_pp_samples=200)
axes[0].set_title("wind_onshore_mw (Gamma GLM)")
az.plot_ppc(solar_hurdle_ppc_idata, var_names=["solar_mw"], ax=axes[1], num_pp_samples=200)
axes[1].set_title("solar_mw (hurdle-Gamma)")
fig.suptitle("Posterior predictive check (original MW units)")
fig.tight_layout()

ppc_path = PROCESSED_DIR / "generation_model_ppc.png"
fig.savefig(ppc_path, dpi=130)
plt.close(fig)
print(f"[generation_model] Saved plot -> {ppc_path}")

frac_zero_ppc = float((solar_hurdle_ppc_y == 0).mean())
print(f"[generation_model] solar_mw posterior predictive zero-mass: {frac_zero_ppc:.1%} "
      f"(observed: {(solar_y == 0).mean():.1%})")

Sampling: [y_obs]


[generation_model] Saved plot -> d:\weather-energy-analytics\data\processed\generation_model_ppc.png
[generation_model] solar_mw posterior predictive zero-mass: 26.4% (observed: 26.3%)


## 8. Formal model comparison: PSIS-LOO vs. a Normal-likelihood baseline

Not just visual: `az.loo` (Pareto-smoothed importance-sampling leave-one-out cross-validation, PSIS-LOO-CV) against a quick Normal-likelihood baseline -- the wrong-but-simpler model this analysis would have shipped without the EDA in Section 2. That baseline is fit here *only* to be discredited by this comparison (it isn't diagnosed on its own merits, since Sections 2 and 7 already show why it's the wrong likelihood for both targets), reported as an ELPD difference -- the same method used in the Bayesian transport-delay group project, so this is consistent methodology across the portfolio rather than a one-off.

In [11]:
def fit_normal_baseline(df: pd.DataFrame, x_cols: list[str], y_col: str, model_name: str) -> dict:
    """Minimal Normal-likelihood baseline -- fit only to quantify, via PSIS-LOO
    below, how much worse it is than the corrected model above."""
    sub = df[x_cols + [y_col]].dropna()
    X = sub[x_cols].to_numpy(dtype=float)
    y = sub[y_col].to_numpy(dtype=float)
    x_mean, x_std = X.mean(axis=0), X.std(axis=0)
    Xz = (X - x_mean) / x_std

    print(f"[generation_model] Fitting Normal baseline: {y_col} ~ {' + '.join(x_cols)} ...")
    with pm.Model(coords={"predictor": x_cols}) as model:
        intercept = pm.Normal("intercept", mu=y.mean(), sigma=y.std())
        beta = pm.Normal("beta", mu=0, sigma=y.std(), dims="predictor")
        sigma = pm.HalfNormal("sigma", sigma=y.std())
        mu = intercept + pm.math.dot(Xz, beta)
        pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y)
        idata = pm.sample(
            draws=N_DRAWS, tune=N_TUNE, chains=N_CHAINS,
            target_accept=0.9, random_seed=RANDOM_SEED, progressbar=False,
        )
        pm.compute_log_likelihood(idata, extend_inferencedata=True)
    return {"model": model, "idata": idata}


wind_normal_baseline = fit_normal_baseline(joined, ["wind_speed_kmh"], "wind_onshore_mw", "wind_onshore")
solar_normal_baseline = fit_normal_baseline(joined, ["temperature_c", "cloud_cover_pct"], "solar_mw", "solar")

pm.compute_log_likelihood(wind_gamma_fit["idata"], model=wind_gamma_fit["model"], extend_inferencedata=True)
pm.compute_log_likelihood(solar_hurdle_fit["idata"], model=solar_hurdle_fit["model"], extend_inferencedata=True)

wind_compare = az.compare({"normal_baseline": wind_normal_baseline["idata"], "gamma": wind_gamma_fit["idata"]})
print("\n[generation_model] wind: PSIS-LOO comparison (Normal baseline vs. Gamma GLM):")
print(wind_compare)

solar_compare = az.compare({"normal_baseline": solar_normal_baseline["idata"], "hurdle_gamma": solar_hurdle_fit["idata"]})
print("\n[generation_model] solar: PSIS-LOO comparison (Normal baseline vs. hurdle-Gamma):")
print(solar_compare)

loo_compare_path = PROCESSED_DIR / "generation_model_loo_compare.csv"
pd.concat(
    [wind_compare.assign(target="wind_onshore"), solar_compare.assign(target="solar")], axis=0
).to_csv(loo_compare_path)
print(f"\n[generation_model] Saved -> {loo_compare_path}")

[generation_model] Fitting Normal baseline: wind_onshore_mw ~ wind_speed_kmh ...


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [intercept, beta, sigma]
Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 3351 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


Output()

[generation_model] Fitting Normal baseline: solar_mw ~ temperature_c + cloud_cover_pct ...


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [intercept, beta, sigma]
Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 3113 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


Output()

Output()

Output()


[generation_model] wind: PSIS-LOO comparison (Normal baseline vs. Gamma GLM):
                 rank      elpd_loo     p_loo  elpd_diff    weight         se  \
gamma               0 -21426.093448  2.856361    0.00000  0.897732  37.258485   
normal_baseline     1 -21698.972988  3.178160  272.87954  0.102268  34.898397   

                       dse  warning scale  
gamma             0.000000    False   log  
normal_baseline  27.196053    False   log  

[generation_model] solar: PSIS-LOO comparison (Normal baseline vs. hurdle-Gamma):
                 rank      elpd_loo     p_loo    elpd_diff    weight  \
hurdle_gamma        0 -17832.722318  5.832696     0.000000  0.804422   
normal_baseline     1 -23839.658770  3.560018  6006.936452  0.195578   

                         se         dse  warning scale  
hurdle_gamma     217.030951    0.000000    False   log  
normal_baseline   26.089520  210.228573    False   log  

[generation_model] Saved -> d:\weather-energy-analytics\data\processed\ge

## 9. Best-case vs. forecast-driven MAE gap, and price-weighted cost

Same logic throughout this notebook: best case predicts on the actual weather used to fit the model (the error floor), forecast-driven predicts on Open-Meteo's forecast weather for the same hours. Point prediction per posterior draw is each model's posterior mean: `mu` for the Gamma GLM (wind), `E[Y] = p_nonzero * mu_positive` for the hurdle-Gamma (solar).

In [12]:
def mae_draws(y_true: np.ndarray, pred_draws: np.ndarray) -> np.ndarray:
    """pred_draws: (chain, draw, n_obs) -> MAE per (chain, draw), shape (chain, draw)."""
    return np.abs(y_true[None, None, :] - pred_draws).mean(axis=-1)


def hdi_report(draws: np.ndarray, label: str) -> dict:
    flat = draws.flatten()
    lo, hi = az.hdi(flat, hdi_prob=HDI_PROB)
    print(f"[generation_model]   {label}: mean={flat.mean():,.2f}, {HDI_PROB:.0%} CI [{lo:,.2f}, {hi:,.2f}]")
    return {"mean": float(flat.mean()), "hdi_low": float(lo), "hdi_high": float(hi)}


def predict_mean_gamma(fit: dict, X_new: np.ndarray) -> np.ndarray:
    """Posterior draws of the Gamma GLM's mean mu at design matrix X_new
    (n_obs, n_predictors). Returns shape (chain, draw, n_obs)."""
    Xz_new = (X_new - fit["x_mean"]) / fit["x_std"]
    b0 = fit["idata"].posterior["b0"].values
    beta = fit["idata"].posterior["beta"].values
    eta = b0[..., None] + np.einsum("cdk,nk->cdn", beta, Xz_new)
    return np.exp(eta)


def predict_mean_hurdle(fit: dict, X_new: np.ndarray) -> np.ndarray:
    """Posterior draws of the hurdle-Gamma's mean E[Y] = p_nonzero * mu_positive
    at design matrix X_new. Returns shape (chain, draw, n_obs)."""
    Xz_new = (X_new - fit["x_mean"]) / fit["x_std"]
    g0 = fit["idata"].posterior["g0"].values
    gamma_beta = fit["idata"].posterior["gamma_beta"].values
    logit_p = g0[..., None] + np.einsum("cdk,nk->cdn", gamma_beta, Xz_new)
    p = 1 / (1 + np.exp(-logit_p))

    b0 = fit["idata"].posterior["b0"].values
    beta = fit["idata"].posterior["beta"].values
    mu = np.exp(b0[..., None] + np.einsum("cdk,nk->cdn", beta, Xz_new))
    return p * mu


results_rows = []
gap_summaries = {}

for fit, model_name, predict_fn in [
    (wind_gamma_fit, "wind_onshore", predict_mean_gamma),
    (solar_hurdle_fit, "solar", predict_mean_hurdle),
]:
    x_cols = fit["x_cols"]
    y_col = fit["y_col"]

    print(f"\n[generation_model] {model_name} -- best case (actual weather):")
    X_best = fit["sub"][x_cols].to_numpy(dtype=float)
    y_best = fit["sub"][y_col].to_numpy(dtype=float)
    pred_best = predict_fn(fit, X_best)
    mae_best_draws = mae_draws(y_best, pred_best)
    best_summary = hdi_report(mae_best_draws, f"{model_name} best-case MAE")

    print(f"[generation_model] {model_name} -- forecast-driven (Open-Meteo forecast weather):")
    fcst_merged = pd.merge(
        joined[["timestamp", y_col]],
        weather_fcst[["timestamp"] + x_cols],
        on="timestamp", how="inner",
    ).dropna(subset=[y_col] + x_cols)
    if fcst_merged.empty:
        raise RuntimeError(f"No overlapping timestamps between weather_energy_joined and weather_forecast for {model_name}.")
    X_fcst = fcst_merged[x_cols].to_numpy(dtype=float)
    y_fcst = fcst_merged[y_col].to_numpy(dtype=float)
    pred_fcst = predict_fn(fit, X_fcst)
    mae_fcst_draws = mae_draws(y_fcst, pred_fcst)
    fcst_summary = hdi_report(mae_fcst_draws, f"{model_name} forecast-driven MAE")

    gap_draws = mae_fcst_draws - mae_best_draws
    hdi_report(gap_draws, f"{model_name} forecast-error gap (forecast-driven minus best-case)")
    gap_summaries[model_name] = {
        "gap_draws": gap_draws, "fcst_merged": fcst_merged,
        "pred_fcst": pred_fcst, "y_fcst": y_fcst,
    }

    results_rows.append({"model": model_name, "scenario": "best_case", **best_summary, "n_obs": len(fit["sub"])})
    results_rows.append({"model": model_name, "scenario": "forecast_driven", **fcst_summary, "n_obs": len(fcst_merged)})

results = pd.DataFrame(results_rows)
results


[generation_model] wind_onshore -- best case (actual weather):
[generation_model]   wind_onshore best-case MAE: mean=4,455.45, 94% CI [4,422.53, 4,500.42]
[generation_model] wind_onshore -- forecast-driven (Open-Meteo forecast weather):
[generation_model]   wind_onshore forecast-driven MAE: mean=4,911.41, 94% CI [4,836.07, 4,991.16]
[generation_model]   wind_onshore forecast-error gap (forecast-driven minus best-case): mean=455.96, 94% CI [413.62, 500.07]

[generation_model] solar -- best case (actual weather):
[generation_model]   solar best-case MAE: mean=12,803.18, 94% CI [12,641.35, 12,991.19]
[generation_model] solar -- forecast-driven (Open-Meteo forecast weather):
[generation_model]   solar forecast-driven MAE: mean=14,390.54, 94% CI [14,092.64, 14,732.33]
[generation_model]   solar forecast-error gap (forecast-driven minus best-case): mean=1,587.36, 94% CI [1,317.48, 1,881.53]


,model,scenario,mean,hdi_low,hdi_high,n_obs
0,wind_onshore,best_case,4455.447790,4422.529930,4500.416507,2157
1,wind_onshore,forecast_driven,4911.408131,4836.073098,4991.159455,2157
2,solar,best_case,12803.176647,12641.346988,12991.187182,2157
3,solar,forecast_driven,14390.536890,14092.639087,14732.326734,2157


In [13]:
if not have_price:
    print("[generation_model] Skipping price-weighted cost: data/raw/energy_prices.parquet not found. Run ingest_price.py first.")
    cost_rows = []
else:
    cost_rows = []
    for model_name, info in gap_summaries.items():
        priced = info["fcst_merged"].merge(prices, on="timestamp", how="left")
        mask = priced["price_eur_mwh"].notna().to_numpy()
        if not mask.any():
            print(f"[generation_model]   {model_name}: no overlapping timestamps with energy_prices.parquet -- skipping.")
            continue
        price_vec = priced.loc[mask, "price_eur_mwh"].to_numpy(dtype=float)
        y_fcst_masked = info["y_fcst"][mask]
        pred_fcst_masked = info["pred_fcst"][:, :, mask]
        abs_error = np.abs(y_fcst_masked[None, None, :] - pred_fcst_masked)
        cost_draws = (abs_error * price_vec[None, None, :]).sum(axis=-1)
        print(f"\n[generation_model] {model_name} -- price-weighted illustrative imbalance cost ({int(mask.sum()):,} priced hours):")
        summary = hdi_report(cost_draws, f"{model_name} imbalance-equivalent cost (EUR)")
        cost_rows.append({"model": model_name, **summary, "n_priced_hours": int(mask.sum())})

cost_df = pd.DataFrame(cost_rows)

fig, ax = plt.subplots(figsize=(7, 4))
models_list = ["wind_onshore", "solar"]
x = np.arange(len(models_list))
width = 0.35


def _err(model, scenario):
    row = results[(results["model"] == model) & (results["scenario"] == scenario)].iloc[0]
    return row["mean"], row["mean"] - row["hdi_low"], row["hdi_high"] - row["mean"]


best_means, best_lo, best_hi = zip(*[_err(m, "best_case") for m in models_list])
fcst_means, fcst_lo, fcst_hi = zip(*[_err(m, "forecast_driven") for m in models_list])

ax.bar(x - width / 2, best_means, width, yerr=[best_lo, best_hi], capsize=4, label="Best case (actual weather)", color="#2c6e9e")
ax.bar(x + width / 2, fcst_means, width, yerr=[fcst_lo, fcst_hi], capsize=4, label="Forecast-driven (forecast weather)", color="#c0392b")
ax.set_xticks(x)
ax.set_xticklabels(models_list)
ax.set_ylabel("MAE (MW)")
ax.set_title(f"Generation model MAE: best case vs. forecast-driven\n(posterior mean, {HDI_PROB:.0%} credible interval)")
ax.legend()
fig.tight_layout()

mae_plot_path = PROCESSED_DIR / "generation_model_mae.png"
fig.savefig(mae_plot_path, dpi=130)
plt.close(fig)
print(f"[generation_model] Saved plot -> {mae_plot_path}")

cost_df


[generation_model] wind_onshore -- price-weighted illustrative imbalance cost (2,123 priced hours):
[generation_model]   wind_onshore imbalance-equivalent cost (EUR): mean=1,210,873,219.84, 94% CI [1,193,745,654.49, 1,230,872,891.63]

[generation_model] solar -- price-weighted illustrative imbalance cost (2,123 priced hours):
[generation_model]   solar imbalance-equivalent cost (EUR): mean=3,742,474,414.25, 94% CI [3,537,066,895.40, 3,972,544,122.53]
[generation_model] Saved plot -> d:\weather-energy-analytics\data\processed\generation_model_mae.png


,model,mean,hdi_low,hdi_high,n_priced_hours
0,wind_onshore,1.210873e+09,1.193746e+09,1.230873e+09,2123
1,solar,3.742474e+09,3.537067e+09,3.972544e+09,2123


## 10. Save results

- `data/processed/generation_model_accuracy.csv` -- model, scenario, MAE posterior mean and 94% credible interval, n_obs.
- `data/processed/generation_model_coefficients.csv` -- fitted coefficient posterior mean and 94% credible interval (log/logit-link scale), per model/term.
- `data/processed/generation_model_diagnostics.csv` -- R-hat, ESS (bulk/tail) per parameter, both models.
- `data/processed/generation_model_gap_summary.csv` -- the forecast-error MAE gap and (if available) the price-weighted imbalance cost, posterior mean and 94% credible interval, per model.
- `data/processed/generation_model_loo_compare.csv` -- PSIS-LOO `az.compare` tables (Normal baseline vs. corrected model), both targets.
- `data/processed/generation_model_summary.txt` -- human-readable version of all of the above.
- `data/processed/generation_model_eda_marginals.png`, `generation_model_prior_predictive.png`, `generation_model_trace_wind.png`, `generation_model_trace_solar.png`, `generation_model_ppc.png`, `generation_model_mae.png` -- the plots from Sections 2-9.

In [14]:
accuracy_path = PROCESSED_DIR / "generation_model_accuracy.csv"
results.to_csv(accuracy_path, index=False)
print(f"[generation_model] Wrote {accuracy_path}")

coef_rows = []
for fit, model_name, param_names in [
    (wind_gamma_fit, "wind_onshore_gamma", ["b0", "beta", "alpha"]),
    (solar_hurdle_fit, "solar_hurdle_gamma", ["g0", "gamma_beta", "b0", "beta", "alpha"]),
]:
    for pname in param_names:
        draws = fit["idata"].posterior[pname].values
        if draws.ndim == 2:  # scalar parameter: (chain, draw)
            lo, hi = az.hdi(draws.flatten(), hdi_prob=HDI_PROB)
            coef_rows.append({
                "model": model_name, "term": pname,
                "mean": float(draws.mean()), "hdi_low": float(lo), "hdi_high": float(hi),
            })
        else:  # vector parameter over "predictor": (chain, draw, n_predictors)
            for i, col in enumerate(fit["x_cols"]):
                d = draws[..., i]
                lo, hi = az.hdi(d.flatten(), hdi_prob=HDI_PROB)
                coef_rows.append({
                    "model": model_name, "term": f"{pname}[{col}]",
                    "mean": float(d.mean()), "hdi_low": float(lo), "hdi_high": float(hi),
                })

coef_df = pd.DataFrame(coef_rows)
coef_path = PROCESSED_DIR / "generation_model_coefficients.csv"
coef_df.to_csv(coef_path, index=False)
print(f"[generation_model] Wrote {coef_path}")
print(coef_df.round(4).to_string(index=False))

diag_df = pd.concat(
    [
        wind_gamma_summary.reset_index().rename(columns={"index": "parameter"}).assign(model="wind_onshore_gamma"),
        solar_hurdle_summary.reset_index().rename(columns={"index": "parameter"}).assign(model="solar_hurdle_gamma"),
    ],
    ignore_index=True,
)
diag_path = PROCESSED_DIR / "generation_model_diagnostics.csv"
diag_df.to_csv(diag_path, index=False)
print(f"[generation_model] Wrote {diag_path}")

gap_rows = []
for model_name, info in gap_summaries.items():
    flat = info["gap_draws"].flatten()
    lo, hi = az.hdi(flat, hdi_prob=HDI_PROB)
    gap_rows.append({"model": model_name, "metric": "mae_gap_mw", "mean": float(flat.mean()), "hdi_low": float(lo), "hdi_high": float(hi)})
if not cost_df.empty:
    for _, row in cost_df.iterrows():
        gap_rows.append({"model": row["model"], "metric": "imbalance_cost_eur", "mean": row["mean"], "hdi_low": row["hdi_low"], "hdi_high": row["hdi_high"]})
gap_summary_df = pd.DataFrame(gap_rows)
gap_summary_path = PROCESSED_DIR / "generation_model_gap_summary.csv"
gap_summary_df.to_csv(gap_summary_path, index=False)
print(f"[generation_model] Wrote {gap_summary_path}")

summary_lines = [
    "generation_model -- Gamma GLM (wind) / hurdle-Gamma (solar), chosen from marginal-distribution",
    "EDA per the correct Bayesian workflow order (EDA -> likelihood choice -> prior predictive check ->",
    "fit -> diagnostics -> posterior predictive check -> formal model comparison).",
    f"MCMC: {N_CHAINS} chains x ({N_TUNE} tune + {N_DRAWS} draws), target_accept=0.9, seed={RANDOM_SEED}",
    "",
    "Convergence diagnostics:",
    f"  All R-hat within 0.01 of 1.00, ESS(bulk) >= 400, 0 divergences: {diag_ok}",
    "  Full table -> generation_model_diagnostics.csv",
    "",
    "PSIS-LOO model comparison vs. a Normal-likelihood baseline (the wrong likelihood this analysis",
    "would otherwise have shipped -- see Section 2's EDA and Section 7's posterior predictive check",
    "for why); full az.compare tables -> generation_model_loo_compare.csv:",
]
for target_name, compare_df in [("wind_onshore", wind_compare), ("solar", solar_compare)]:
    summary_lines.append(f"  {target_name}:")
    summary_lines.append("    " + compare_df.to_string().replace("\n", "\n    "))

summary_lines += ["", f"Fitted coefficients (log/logit-link scale), posterior mean and {HDI_PROB:.0%} credible interval:"]
for _, row in coef_df.iterrows():
    summary_lines.append(f"  {row['model']:<20s} {row['term']:<25s} {row['mean']:>10.4f}  [{row['hdi_low']:.4f}, {row['hdi_high']:.4f}]")

summary_lines += ["", f"Accuracy cost of weather forecast error (forecast_driven - best_case MAE), posterior mean and {HDI_PROB:.0%} CI:"]
for model_name, info in gap_summaries.items():
    flat = info["gap_draws"].flatten()
    lo, hi = az.hdi(flat, hdi_prob=HDI_PROB)
    summary_lines.append(f"  {model_name}: +{flat.mean():,.2f} MW  [{lo:,.2f}, {hi:,.2f}]")

if cost_df.empty:
    summary_lines += ["", "Price-weighted imbalance-equivalent cost: SKIPPED (data/raw/energy_prices.parquet not found)."]
else:
    summary_lines += [
        "",
        "Illustrative imbalance-equivalent cost (forecast-driven scenario, price-weighted by the actual "
        "hourly day-ahead price -- NOT a real settlement/trading estimate; real imbalance settlement uses "
        "signed error against a dedicated imbalance price, not absolute error against the day-ahead price):",
    ]
    for _, row in cost_df.iterrows():
        summary_lines.append(f"  {row['model']}: EUR {row['mean']:,.0f}  [{row['hdi_low']:,.0f}, {row['hdi_high']:,.0f}]  (n={int(row['n_priced_hours']):,} priced hours)")

summary_path = PROCESSED_DIR / "generation_model_summary.txt"
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")
print(f"[generation_model] Wrote {summary_path}")
print()
print("\n".join(summary_lines))

[generation_model] Wrote d:\weather-energy-analytics\data\processed\generation_model_accuracy.csv
[generation_model] Wrote d:\weather-energy-analytics\data\processed\generation_model_coefficients.csv
             model                        term   mean  hdi_low  hdi_high
wind_onshore_gamma                          b0 9.1015   9.0762    9.1293
wind_onshore_gamma        beta[wind_speed_kmh] 0.4320   0.4078    0.4598
wind_onshore_gamma                       alpha 2.3784   2.2441    2.4958
solar_hurdle_gamma                          g0 1.3673   1.2499    1.4898
solar_hurdle_gamma   gamma_beta[temperature_c] 1.2919   1.1437    1.4259
solar_hurdle_gamma gamma_beta[cloud_cover_pct] 0.2662   0.1632    0.3721
solar_hurdle_gamma                          b0 9.7985   9.7285    9.8678
solar_hurdle_gamma         beta[temperature_c] 0.3740   0.2962    0.4542
solar_hurdle_gamma       beta[cloud_cover_pct] 0.0352  -0.0490    0.1005
solar_hurdle_gamma                       alpha 0.4459   0.4202    0.46